# 05.1 — Vanilla RNN from Scratch

**Problem it solves:** Sequences have memory — what came before matters. An RNN processes tokens one at a time, carrying a hidden state that accumulates context.

**The recurrence:** h_t = tanh(W_hh × h_{t-1} + W_xh × x_t + b)

**Why build it in numpy:** You must understand BPTT (backprop through time) and the vanishing gradient problem before you can appreciate LSTMs.

---

In [ ]:
import numpy as np

class VanillaRNN:
    """
    Vanilla RNN with Backpropagation Through Time (BPTT).
    
    Forward pass:
      h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
      y_t = W_hy @ h_t + b_y
    """
    
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.lr = lr
        
        # Xavier initialization
        scale = 0.01
        self.W_xh = np.random.randn(hidden_size, input_size)  * scale
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale
        self.W_hy = np.random.randn(output_size, hidden_size) * scale
        self.b_h  = np.zeros((hidden_size, 1))
        self.b_y  = np.zeros((output_size, 1))
    
    def forward(self, inputs, h_prev):
        """
        inputs: list of input vectors (each shape [input_size, 1])
        h_prev: initial hidden state
        Returns: list of outputs, list of hidden states, cache for BPTT
        """
        cache = {'inputs': inputs, 'hs': {-1: h_prev}, 'ys': {}, 'ps': {}}
        h = h_prev
        outputs = []
        
        for t, x in enumerate(inputs):
            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            y = self.W_hy @ h + self.b_y
            # Softmax
            p = np.exp(y - y.max())
            p = p / p.sum()
            
            cache['hs'][t] = h
            cache['ys'][t] = y
            cache['ps'][t] = p
            outputs.append(p)
        
        return outputs, h, cache
    
    def backward(self, cache, targets):
        """
        BPTT: unroll gradients through time.
        targets: list of integer indices (true next tokens)
        """
        T = len(targets)
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        dW_hy = np.zeros_like(self.W_hy)
        db_h  = np.zeros_like(self.b_h)
        db_y  = np.zeros_like(self.b_y)
        
        dh_next = np.zeros_like(cache['hs'][0])
        total_loss = 0.0
        
        for t in reversed(range(T)):
            p = cache['ps'][t]
            h = cache['hs'][t]
            h_prev = cache['hs'][t-1]
            x = cache['inputs'][t]
            
            # Cross-entropy loss
            total_loss -= np.log(p[targets[t], 0] + 1e-10)
            
            # Gradient of softmax + cross-entropy
            dy = p.copy()
            dy[targets[t]] -= 1  # subtract 1 at true class
            
            dW_hy += dy @ h.T
            db_y  += dy
            
            # Backprop into h
            dh = self.W_hy.T @ dy + dh_next
            
            # Backprop through tanh: dtanh = (1 - tanh²) * upstream
            dtanh = (1 - h**2) * dh
            
            db_h  += dtanh
            dW_xh += dtanh @ x.T
            dW_hh += dtanh @ h_prev.T
            
            dh_next = self.W_hh.T @ dtanh
        
        # Gradient clipping: prevent exploding gradients
        for dparam in [dW_xh, dW_hh, dW_hy, db_h, db_y]:
            np.clip(dparam, -5, 5, out=dparam)
        
        return total_loss, dW_xh, dW_hh, dW_hy, db_h, db_y
    
    def update(self, dW_xh, dW_hh, dW_hy, db_h, db_y):
        self.W_xh -= self.lr * dW_xh
        self.W_hh -= self.lr * dW_hh
        self.W_hy -= self.lr * dW_hy
        self.b_h  -= self.lr * db_h
        self.b_y  -= self.lr * db_y
    
    def sample(self, h, seed_idx, n_chars, vocab_size):
        """Sample from the RNN character by character."""
        x = np.zeros((vocab_size, 1))
        x[seed_idx] = 1
        indices = []
        
        for _ in range(n_chars):
            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            y = self.W_hy @ h + self.b_y
            p = np.exp(y - y.max())
            p = p / p.sum()
            idx = np.random.choice(vocab_size, p=p.ravel())
            x = np.zeros((vocab_size, 1))
            x[idx] = 1
            indices.append(idx)
        
        return indices

print("VanillaRNN class defined.")
print("Architecture:")
print("  Input  x_t  → [input_size, 1]")
print("  Hidden h_t  = tanh(W_xh@x_t + W_hh@h_{t-1} + b_h)")
print("  Output y_t  = W_hy@h_t + b_y")
print("  Softmax → probability distribution over vocab")

In [ ]:
# Train a character-level language model

text = "the quick brown fox jumps over the lazy dog " * 5

# Build character vocabulary
chars = sorted(set(text))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
V = len(chars)

print(f"Corpus: {len(text)} chars, vocab: {V} unique chars: {repr(chars)}")

rnn = VanillaRNN(input_size=V, hidden_size=64, output_size=V, lr=0.01)
seq_len = 25

losses = []
h = np.zeros((64, 1))

for iteration in range(500):
    start = (iteration * seq_len) % (len(text) - seq_len - 1)
    
    inputs = [
        (lambda x: (x.__setitem__((char2idx[text[start+t]], 0), 1), x)[1])(np.zeros((V,1)))
        for t in range(seq_len)
    ]
    targets = [char2idx[text[start+t+1]] for t in range(seq_len)]
    
    _, h, cache = rnn.forward(inputs, h)
    loss, *grads = rnn.backward(cache, targets)
    rnn.update(*grads)
    h = np.zeros((64, 1))  # reset hidden state between chunks
    
    if iteration % 100 == 0:
        losses.append(loss)
        # Sample
        seed = char2idx['t']
        sampled = rnn.sample(np.zeros((64,1)), seed, 50, V)
        generated = 't' + ''.join(idx2char[i] for i in sampled)
        print(f"Iter {iteration:4d}: loss={loss:.2f}  sample: {generated!r}")

In [ ]:
# The vanishing gradient problem — the core reason LSTMs were invented

print("=== VANISHING GRADIENT DEMONSTRATION ===")
print()
print("In BPTT, gradients flow through T time steps:")
print("  dL/dh_0 = (dL/dh_T) × ∏_{t=1}^{T} dh_t/dh_{t-1}")
print()
print("Each factor = diag(1 - h_t²) × W_hh")
print("If the spectral radius of W_hh < 1, gradients shrink exponentially.")
print()

# Simulate gradient magnitude over time steps
def simulate_gradient_flow(W_hh_init, n_steps=50):
    """Track how gradient magnitude changes over time steps."""
    grad_magnitudes = [1.0]
    grad = 1.0
    
    for t in range(n_steps):
        # At each step, multiply by the Jacobian norm
        # Simplified: use spectral radius of W_hh
        spectral_radius = np.max(np.abs(np.linalg.eigvals(W_hh_init)))
        grad *= spectral_radius * 0.5  # 0.5 accounts for tanh saturation
        grad_magnitudes.append(grad)
    
    return grad_magnitudes

H = 10
for scale, label in [(0.1, 'small init (vanishing)'), 
                      (0.5, 'medium init'), 
                      (2.0, 'large init (exploding)')]:
    W = np.random.randn(H, H) * scale
    mags = simulate_gradient_flow(W)
    print(f"{label:35} gradients at step 50: {mags[50]:.2e}")

print()
print("Vanishing: grad → 0, long-range dependencies lost")
print("Exploding: grad → ∞, training diverges (fixed with gradient clipping)")
print("LSTM fix: add cell state with additive updates — gradient highway")

## Summary

**What RNN teaches you:**
- Sequential processing with memory
- BPTT: same backprop, applied through time
- Gradient clipping: cheap fix for exploding gradients
- Vanishing gradients: NOT fixable by clipping — need LSTM/GRU

**What breaks RNNs:**
- Long-range dependencies: gradient signal from 50 steps ago is ≈0
- Sequential processing prevents parallelism (slow on GPU)
- Training instability with tanh saturation

---
**Next:** 05.2 — LSTM: solving vanishing gradients with gates